# Simulation of LUCJ circuits

In [ ]:
from typing import Optional
import torch

import ffsim
import matplotlib.pyplot as plt; plt.rcParams.update({"font.family": "serif", "font.size": 12})
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf

import qiskit
from qiskit import qasm2
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import Statevector, SparsePauliOp, DensityMatrix

import cirq
from cirq.contrib import qasm_import
import recirq
import quimb as qu
import quimb.tensor as qtn

## Tensor network simulator

In [ ]:
def simulate(
    circuit: cirq.Circuit,
    verbose: bool = False,
    seed: Optional[int] = None,
    backend: str = "cpu",
    max_bond: Optional[int] = None,
    cutoff: float = 0.0,
) -> qtn.MatrixProductState:
    max_bonds = []
    rng = np.random.RandomState(seed)

    qubits_to_indices = {q: i for i, q in enumerate(sorted(circuit.all_qubits()))}
    nqubits = len(qubits_to_indices)

    mps = qtn.MPS_computational_state("0" * nqubits, dtype="float64", cyclic=False)

    if backend == "gpu":
        for tensor in mps.tensors:
            tensor.modify(
                apply=lambda x: torch.tensor(x, dtype=torch.complex64, device="cuda")
            )

    num_ops = len(list(circuit.all_operations()))
    for i, op in enumerate(circuit.all_operations()):
        qubit_indices = [qubits_to_indices[q] for q in op.qubits]
        if cirq.has_unitary(op):
            to_apply = qu.qarray(cirq.unitary(op))
        elif cirq.has_mixture(op):
            ps = []
            ops = []
            for (p, o) in cirq.mixture(op):
                ps.append(p)
                ops.append(o)
            op = ops[rng.choice(range(len(ops)), p=ps)]
            to_apply = qu.qarray(op)
        else:
            raise ValueError(f"Cannot apply operation {op}")

        if backend == "gpu":
            to_apply = torch.tensor(to_apply, dtype=torch.complex64, device="cuda")

        mps.gate_(
            to_apply,
            qubit_indices,
            contract="swap+split",
            max_bond=max_bond,
            cutoff=cutoff,
        )
        mps.compress()
        if verbose:
            max_bonds.append(mps.max_bond())
            print(f"\rOp {i + 1} / {num_ops}, max bond = {mps.max_bond()}", end="")

    if verbose:
        return mps, max_bonds
    return mps

In [ ]:
from recirq.beyond_classical import generate_boixo_2018_beyond_classical_v2

In [ ]:
circuit = generate_boixo_2018_beyond_classical_v2(
    cirq.GridQubit.rect(nx, nx),
    cz_depth=24,
    seed=1,
)
len(list(circuit.all_operations()))

In [ ]:
mps, max_bonds = simulate(circuit, verbose=True)

In [ ]:
plt.semilogy(max_bonds, "--o")
plt.axhline(2 ** (nx * nx / 2))

## Parameters

In [ ]:
nx = 4
atom: str = "H"
natoms: int = nx ** 2 // 2
nlayers: int = 20
local: bool = True

damping: float = 0.001325
cutoff: float = 1e-6
nthreads: int = 24

In [ ]:
3 / 4 * (1 - np.exp(-damping))

## Problem defintion

### Molecule

In [ ]:
def generate_linear_geometry(atom: str, natoms: int, atomic_distance: float = 1.0) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"{atom} 0 0 {i * atomic_distance}" for i in range(natoms)])

In [ ]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=generate_linear_geometry(atom, natoms),
    basis="sto-6g",
)

# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
# reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

## Circuit construction

In [ ]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

In [ ]:
coupling_map = CouplingMap.from_grid(
    num_rows=int(np.ceil(np.sqrt(2 * norb))),
    num_columns=int(np.ceil(np.sqrt(2 * norb)))
)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

In [ ]:
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p) for p in range(norb)]  # None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

In [ ]:
# Create pass manager
try:
    pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=norb,
        connectivity="square",
        interaction_pairs=None if not local else (pairs_aa, pairs_ab),
        optimization_level=3,
    )
    print("Unable to generate ffsim pass manager")
except RuntimeError:
    pass_manager = None

print("pairs_aa:", pairs_aa)
print("pairs_ab:", pairs_ab)

In [ ]:
# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=nlayers,
    interaction_pairs=None if not local else (pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

qubits = qiskit.QuantumRegister(2 * norb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)

In [ ]:
if pass_manager is not None:
    compiled = pass_manager.run(circuit)
else:
    compiled = qiskit.transpile(circuit, backend=backend, optimization_level=3)

In [ ]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

In [ ]:
compiled.draw(fold=-1)

In [ ]:
compiled_cirq = cirq.contrib.qasm_import.circuit_from_qasm(qasm2.dumps(compiled))
compiled_cirq

In [ ]:
mps_lucj, max_bonds_lucj = simulate(compiled_cirq, verbose=True)

In [ ]:
plt.plot(max_bonds_lucj, "--o", label="LUCJ")
plt.plot(max_bonds, "--o", label="RCS")

plt.legend()

plt.xlabel("Gate index")
plt.ylabel(r"$\chi_\text{max}$");

In [ ]:
nlayers_ucj = nlayers // 2

In [ ]:
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=nlayers,
    interaction_pairs=None,
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

qubits = qiskit.QuantumRegister(2 * norb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)

In [ ]:
compiled = qiskit.transpile(circuit, backend=backend, optimization_level=3)

In [ ]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

In [ ]:
compiled_cirq = cirq.contrib.qasm_import.circuit_from_qasm(qasm2.dumps(compiled))
compiled_cirq

In [ ]:
mps_ucj, max_bonds_ucj = simulate(compiled_cirq, verbose=True)

In [ ]:
plt.semilogy(max_bonds_lucj, "--s", mec="black", alpha=0.5, label=f"LUCJ ({compiled.num_qubits} qubits, {nlayers} layers)")
plt.semilogy(max_bonds_ucj, "--s", mec="black", alpha=0.5, label=f"UCJ ({compiled.num_qubits} qubits, {nlayers_ucj} layers)")
plt.semilogy(max_bonds, "--s", mec="black", alpha=0.5, label="RCS")
plt.axhline(2 ** (nx * nx / 2), ls="--", color="black")

plt.legend()

plt.xlabel("Gate index")
plt.ylabel(r"$\chi_\text{max}$");

## Observable definition

In [ ]:
observables = [
    SparsePauliOp("ZZZZ" + "I" * (compiled.num_qubits - 4)),
    SparsePauliOp("I" * (compiled.num_qubits // 2) + "ZZ" + "I" * (compiled.num_qubits // 2 - 2)),
    SparsePauliOp("I" * (compiled.num_qubits - 4) + "ZZZZ")
]

### Exact values

In [ ]:
if compiled.num_qubits <= 20:
    expectation_values_exact = []
    for observable in observables:
        statevector = Statevector(compiled)
        sv_expectation_value = statevector.expectation_value(observable).real
        print(sv_expectation_value)
        expectation_values_exact.append(sv_expectation_value)

### Noisy values

In [ ]:
from qiskit_aer.noise import depolarizing_error
from qiskit.converters import circuit_to_dag, dag_to_circuit

In [ ]:
noisy = compiled.copy_empty_like()
for layer in circuit_to_dag(compiled).layers():
    noisy.compose(dag_to_circuit(layer["graph"]), inplace=True)
    noisy.barrier()
    for qubit in noisy.qubits:
        noisy.append(depolarizing_error(3 / 4 * (1 - np.exp(-damping)), 1), [qubit])  # TODO: Triple check channel rate.
    noisy.barrier()


noisy.draw(fold=-1)

In [ ]:
if noisy.num_qubits <= 10:
    rho = DensityMatrix.from_label("0" * noisy.num_qubits)
    rho = rho.evolve(noisy)

    for observable in observables:
        ev = rho.expectation_value(observable)
        print(ev)

## Heisenberg simulation

In [ ]:
from propaq.datatypes import MajoranaMonomial

from propaq.propagators import MajoranaPropagator, PauliPropagator
from propaq.circuits import MajoranaCircuit, PauliCircuit
from propaq.noise import UniformNoiseModel, truncation
from propaq.noise import TruncationPolicy 

from propaq.datatypes import MajoranaTermSum, PauliTermSum

### Majorana basis

In [ ]:
mc = MajoranaCircuit.from_qiskit(compiled.copy(), n_modes=2 * compiled.num_qubits)

In [ ]:
from propaq import Logger, LogParser

In [ ]:
logger = Logger(filename="propaq.log", log_every=10)

In [ ]:
prop = MajoranaPropagator(
    UniformNoiseModel(damping=damping),
    TruncationPolicy(weight_cutoff=100000000, coeff_cutoff=cutoff),
    n_threads=nthreads,
    progress_bar=True,
    # logger=logger,
)

In [ ]:
results = []
for observable in observables[:]:
    observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
    results.append(prop.expectation_value(observable_mts, mc, fock_state=0))
    print(results[-1].expectation_value)

In [ ]:
parser = LogParser("propaq.log")

In [ ]:
ndiscarded = []
l1discarded = []
for t in parser.truncation_events:
    ndiscarded.append(t.terms_discarded)
    l1discarded.append(t.discarded_coeff_l1)

In [ ]:
plt.semilogy(np.cumsum(ndiscarded), "--o", label="Number of terms discarded")
plt.semilogy(np.cumsum(l1discarded), "--o", label="L1 norm discarded")

plt.xlabel("Truncation event")
plt.grid()
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s), noise damping = {damping}, cutoff = {cutoff}\n{compiled.count_ops()}")
plt.legend();

In [ ]:
np.abs(results[0].expectation_value - expectation_values_exact[0])

In [ ]:
for i, result in enumerate(results):
    plt.semilogy(result.n_terms, "--o", alpha=0.55, label="ZZ " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Gate index")
plt.ylabel("Number of Majorana monomials")
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s)\nnoise damping = {damping}, cutoff = {cutoff}, nthreads = {nthreads}\n{compiled.count_ops()}")
plt.legend();

### Pauli basis

In [ ]:
pc = PauliCircuit.from_qiskit(compiled.copy())

In [ ]:
pauli = PauliPropagator(
    UniformNoiseModel(damping=damping),
    TruncationPolicy(weight_cutoff=100000, coeff_cutoff=cutoff),
    n_threads=16,
    progress_bar=True,
)

In [ ]:
pauli_results = []
for observable in observables:
    observable_mts = PauliTermSum.from_sparse_pauli_op(observable)
    pauli_results.append(pauli.expectation_value(observable_mts, pc, fock_state=0))
    print(pauli_results[-1].expectation_value)

In [ ]:
for i, result in enumerate(pauli_results):
    plt.semilogy(result.n_terms, "--o", alpha=0.55,label="ZZ " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Gate index")
plt.ylabel("Number of Pauli strings")
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s), noise damping = {damping}, cutoff = {cutoff}\n{compiled.count_ops()}")
plt.legend();

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharey=True
)

# Left subplot: Majorana monomials
ax = axes[0]

for i, result in enumerate(results):
    ax.semilogy(
        result.n_terms,
        "--o",
        alpha=0.55,
        label="ZZ " + ",".join(
            map(str, observables[i].to_sparse_list()[0][1])
        )
    )

ax.set_xlabel("Gate index")
ax.set_ylabel("Number of terms")
ax.set_title("Majorana monomials")
ax.grid(True, which="both", linestyle="--", alpha=0.75)
ax.legend()


# Right subplot: Pauli strings
ax = axes[1]

for i, result in enumerate(pauli_results):
    ax.semilogy(
        result.n_terms,
        "--o",
        alpha=0.55,
        label="ZZ " + ",".join(
            map(str, observables[i].to_sparse_list()[0][1])
        )
    )

ax.set_xlabel("Gate index")
ax.set_title("Pauli strings")
ax.grid(True, which="both", linestyle="--", alpha=0.5)
ax.legend()


fig.suptitle(
    f"{natoms} {atom} atom(s), "
    f"{len(compiled.qubits)} qubits, "
    f"{nlayers} LUCJ layer(s), "
    f"noise damping = {damping}, cutoff = {cutoff}\n"
    f"{compiled.count_ops()}"
)

plt.tight_layout()
plt.show()

## Accuracy vs coefficient cutoff threshold

In [ ]:
cutoffs = [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
evals = []
for cutoff in cutoffs:
    print("On cutoff:", cutoff)
    prop = MajoranaPropagator(
        UniformNoiseModel(damping=0.00),
        TruncationPolicy(weight_cutoff=100000, coeff_cutoff=cutoff),
        n_threads=2,
        progress_bar=True,
    )
    oevals = []
    for i, observable in enumerate(observables):
        observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
        mp_expectation_value = prop.expectation_value(observable_mts, mc, fock_state=0).expectation_value
        print(mp_expectation_value, expectation_values_exact[i])
        oevals.append(mp_expectation_value)
    evals.append(oevals)

In [ ]:
evals = np.array(evals).T

errors = []
for i, e in enumerate(evals):
    errors.append(e - expectation_values_exact[i])

errors = np.abs(np.array(errors))

In [ ]:
for i, e in enumerate(errors):
    plt.loglog(cutoffs, e, "--s", ms=9, mec="black", alpha=0.75, label="ZZ " + ",".join(map(str, observables[i].to_sparse_list()[0][1])))

plt.xlabel("Coefficient magnitude cutoff")
plt.ylabel(f"Absolute error")
plt.legend()
plt.title(f"{natoms} {atom} atom(s), {len(compiled.qubits)} qubits, {nlayers} LUCJ layer(s)\n{compiled.count_ops()}");
plt.savefig("error_vs_cutoff.pdf")